# Qwen3.5-4B clinical multi-task LoRA fine-tuning (rank 32, five-fold CV)

This notebook fine-tunes **Qwen/Qwen3.5-4B** on the five clinical multi-task folds in:

`/data/liangz2/openi/midrc/multi_task_CV`

Primary image-to-JSON tasks:

1. `mrale_prediction`
2. `covid_classification`
3. `normality_classification`

The same folds, task definitions, path rewriting, internal validation procedure, and evaluation metrics used in the MedGemma notebook are retained for a fair comparison. Qwen's default thinking mode is disabled when rendering prompts so that supervised targets and held-out outputs remain direct, parseable JSON.

Important interpretation: the COVID label is PCR status in MIDRC but a dataset-derived negative label in Montgomery. Therefore, COVID performance across the combined sources can partly measure source/domain differences and is not a standalone clinical validation of COVID diagnosis.


## 1. Imports and environment check

No installation commands are included. Run this notebook in the already repaired environment. Restart the kernel first if packages were changed earlier in the session.

In [ ]:
import csv
import gc
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from importlib.metadata import version
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset

from huggingface_hub import notebook_login, whoami, is_offline_mode
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

print("Python packages")
for package in ["torch", "transformers", "huggingface-hub", "peft", "accelerate", "pillow"]:
    try:
        print(f"  {package}: {version(package)}")
    except Exception as exc:
        print(f"  {package}: unavailable ({exc})")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
print("Hugging Face offline mode:", is_offline_mode())

## 2. Hugging Face authentication

Authenticate before downloading `Qwen/Qwen3.5-4B`, especially on a shared compute node or when Hugging Face rate limits apply. The token is stored by the Hugging Face client and is not written into this notebook.


In [ ]:
notebook_login()  # Paste your hf_... token into the widget.
identity = whoami()
print("Logged in as:", identity["name"])

## 3. Configuration

The five train/test fold files are read directly from `FOLD_DATA_DIR`. Each fold's training file is divided patient-wise where possible into optimization and internal-validation subsets; the provided test fold remains untouched until final evaluation.

The LoRA rank is 32. Qwen3.5 has a hybrid text architecture, so the notebook discovers all linear layers in the text backbone instead of relying only on conventional attention projection names.


In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
FOLD_DATA_DIR = Path("/data/liangz2/openi/midrc/multi_task_CV")
TRAIN_PATTERN = "multitask_train_fold_{fold}_harmony.jsonl"
TEST_PATTERN = "multitask_test_fold_{fold}_harmony.jsonl"
OUTPUT_ROOT = Path("/data/liangz2/openi/midrc/qwen35_4b_multitask_lora_rank32_cv")

RUN_FOLDS = [0, 1, 2, 3, 4]
SEED = 42
INTERNAL_VALIDATION_FRACTION = 0.10
MAX_LENGTH = 2048

# Qwen3.5 uses 16-pixel vision patches followed by 2x2 spatial merging,
# so one language-model image token represents roughly 32x32 input pixels.
# Cap each CXR at 1,024 visual tokens, leaving at least 1,024 positions for
# instructions and the supervised clinical JSON response.
QWEN_VISION_PIXEL_FACTOR = 16 * 2
MIN_IMAGE_TOKENS = 256
MAX_IMAGE_TOKENS = 1024
MIN_IMAGE_PIXELS = MIN_IMAGE_TOKENS * QWEN_VISION_PIXEL_FACTOR**2
MAX_IMAGE_PIXELS = MAX_IMAGE_TOKENS * QWEN_VISION_PIXEL_FACTOR**2

NUM_TRAIN_EPOCHS = 5
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.05
DATALOADER_NUM_WORKERS = 2

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

RUN_GENERATION_EVAL = True
GENERATION_MAX_NEW_TOKENS = {
    "mrale_prediction": 256,
    "covid_classification": 64,
    "normality_classification": 64,
}

DIRECT_TASKS = {
    "mrale_prediction",
    "covid_classification",
    "normality_classification",
}
AUXILIARY_TASKS = {
    "covid_truthfulness",
    "mrale_truthfulness",
    "normality_truthfulness",
}
TRAIN_TASKS = DIRECT_TASKS | AUXILIARY_TASKS

# Qwen3.5 uses thinking mode by default. Direct JSON supervision and evaluation
# should not include a <think> trace.
DISABLE_THINKING = True

# Rewrite the image paths stored in the local JSONLs to their cloud location.
IMAGE_PATH_REWRITES = {
    "/vf/users/liangz2/openi": "/data/liangz2/openi",
}


def choose_model_dtype():
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability(0)
        return torch.bfloat16 if major >= 8 else torch.float16
    return torch.float32


MODEL_DTYPE = choose_model_dtype()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert 0 < INTERNAL_VALIDATION_FRACTION < 0.5
assert LORA_R == 32
assert MAX_IMAGE_TOKENS < MAX_LENGTH

print("Model:", MODEL_ID)
print("Model dtype:", MODEL_DTYPE)
print("Fold data:", FOLD_DATA_DIR)
print("Output:", OUTPUT_ROOT)
print(
    "Qwen image budget:",
    f"{MIN_IMAGE_TOKENS}-{MAX_IMAGE_TOKENS} visual tokens",
    f"({MIN_IMAGE_PIXELS:,}-{MAX_IMAGE_PIXELS:,} pixels)",
)


## 4. Read, validate, and split the fold records

Internal validation is taken only from each training file. Records sharing the same MIDRC study directory are kept together; all records for a Montgomery image are also kept together. The held-out test file is never used for early stopping or model selection.

In [ ]:
def render_chat_template(processor, messages, add_generation_prompt):
    template_kwargs = {
        "add_generation_prompt": add_generation_prompt,
        "tokenize": False,
    }
    if DISABLE_THINKING:
        template_kwargs["enable_thinking"] = False
    return processor.apply_chat_template(messages, **template_kwargs)


def rewrite_image_path(path):
    path = str(path)
    for old_prefix, new_prefix in IMAGE_PATH_REWRITES.items():
        if path.startswith(old_prefix):
            return new_prefix + path[len(old_prefix):]
    return path


def read_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}:{line_number}: {exc}") from exc
            record["image_path"] = rewrite_image_path(record["image_path"])
            records.append(record)
    return records


def answer_text(record):
    ground_truth = record.get("ground_truth", {})
    if isinstance(ground_truth, dict) and "answer" in ground_truth:
        return str(ground_truth["answer"]).strip()
    for message in reversed(record["messages"]):
        if message.get("role") == "assistant":
            content = message.get("content", "")
            if isinstance(content, str):
                return content.strip()
            for item in content:
                if isinstance(item, dict) and item.get("type") == "text":
                    return str(item["text"]).strip()
    raise ValueError(f"No assistant text in record {record.get('id')}")


def validate_records(records, allowed_tasks, split_name, check_images=True):
    seen_ids = set()
    task_counts = Counter()
    missing_images = []
    for record in records:
        required = {"id", "image_path", "task", "messages"}
        missing_keys = required - set(record)
        if missing_keys:
            raise ValueError(f"{split_name}: {record.get('id')} missing {sorted(missing_keys)}")
        if record["id"] in seen_ids:
            raise ValueError(f"{split_name}: duplicate id {record['id']}")
        seen_ids.add(record["id"])
        if record["task"] not in allowed_tasks:
            raise ValueError(f"{split_name}: unexpected task {record['task']}")
        target = answer_text(record)
        if record["task"] in AUXILIARY_TASKS and target not in {"TRUE", "FALSE"}:
            raise ValueError(f"{split_name}: invalid truthfulness target {target!r}")
        if record["task"] in DIRECT_TASKS:
            parsed = json.loads(target)
            if record["task"] == "covid_classification" and parsed.get("covid_positive") not in {"Yes", "No"}:
                raise ValueError(f"{split_name}: invalid COVID target")
            if record["task"] == "normality_classification" and parsed.get("normal") not in {"Yes", "No"}:
                raise ValueError(f"{split_name}: invalid normality target")
            if record["task"] == "mrale_prediction" and "mRALE Score" not in parsed:
                raise ValueError(f"{split_name}: missing mRALE Score")
        if check_images and not Path(record["image_path"]).is_file():
            missing_images.append(record["image_path"])
        task_counts[record["task"]] += 1
    if missing_images:
        examples = "\n".join(missing_images[:5])
        raise FileNotFoundError(
            f"{split_name}: {len(missing_images)} images are missing. Examples:\n{examples}\n"
            "Update IMAGE_PATH_REWRITES if the cloud image root differs."
        )
    return task_counts


def source_name(record):
    source = str(record.get("meta", record.get("metadata", {})).get("source_dataset", "")).lower()
    if "montgomery" in source or "montgomery" in record["image_path"].lower():
        return "montgomery"
    return "midrc"


def validation_group(record):
    image_path = Path(record["image_path"])
    if source_name(record) == "montgomery":
        return f"montgomery::{image_path}"
    return f"midrc::{image_path.parent}"


def grouped_train_validation_split(records, fraction, seed):
    groups_by_source = defaultdict(lambda: defaultdict(list))
    for record in records:
        groups_by_source[source_name(record)][validation_group(record)].append(record)

    validation_groups = set()
    for source, groups in sorted(groups_by_source.items()):
        keys = sorted(groups)
        rng = random.Random(seed + sum(ord(char) for char in source))
        rng.shuffle(keys)
        n_validation = max(1, round(len(keys) * fraction)) if len(keys) > 1 else 0
        validation_groups.update(keys[:n_validation])

    train_records, validation_records = [], []
    for record in records:
        destination = validation_records if validation_group(record) in validation_groups else train_records
        destination.append(record)

    train_groups = {validation_group(record) for record in train_records}
    validation_group_set = {validation_group(record) for record in validation_records}
    assert train_groups.isdisjoint(validation_group_set)
    return train_records, validation_records


def label_stratum(record):
    task = record["task"]
    target = answer_text(record)
    if task == "covid_classification":
        return json.loads(target)["covid_positive"]
    if task == "normality_classification":
        return json.loads(target)["normal"]
    if task in AUXILIARY_TASKS:
        return target
    return None


def sampled(items, count, rng):
    if not items or count <= 0:
        return []
    if count <= len(items):
        return rng.sample(items, count)
    return list(items) + [rng.choice(items) for _ in range(count - len(items))]


def make_balanced_training_epoch(records, seed):
    """Moderately rebalance tasks with sqrt-frequency targets; balance binary labels within task."""
    by_task = defaultdict(list)
    for record in records:
        by_task[record["task"]].append(record)

    total = len(records)
    weights = {task: math.sqrt(len(items)) for task, items in by_task.items()}
    denominator = sum(weights.values())
    targets = {task: max(1, round(total * weight / denominator)) for task, weight in weights.items()}
    difference = total - sum(targets.values())
    ordered = sorted(targets, key=lambda task: (-weights[task], task))
    cursor = 0
    while difference != 0:
        task = ordered[cursor % len(ordered)]
        change = 1 if difference > 0 else -1
        if targets[task] + change >= 1:
            targets[task] += change
            difference -= change
        cursor += 1

    rng = random.Random(seed)
    result = []
    for task in sorted(by_task):
        items = by_task[task]
        strata = defaultdict(list)
        for item in items:
            strata[label_stratum(item)].append(item)
        if set(strata) <= {None}:
            result.extend(sampled(items, targets[task], rng))
            continue

        labels = sorted(strata, key=str)
        base, remainder = divmod(targets[task], len(labels))
        for index, label in enumerate(labels):
            result.extend(sampled(strata[label], base + (index < remainder), rng))
    rng.shuffle(result)
    assert len(result) == total
    return result


def describe_records(name, records):
    print(f"{name}: {len(records):,}")
    print("  tasks:", dict(sorted(Counter(record["task"] for record in records).items())))
    print("  sources:", dict(sorted(Counter(source_name(record) for record in records).items())))
    binary = Counter((record["task"], label_stratum(record)) for record in records if label_stratum(record) is not None)
    print("  binary labels:", dict(sorted(binary.items(), key=lambda item: str(item[0]))))


def load_fold(fold, check_images=True):
    train_path = FOLD_DATA_DIR / TRAIN_PATTERN.format(fold=fold)
    test_path = FOLD_DATA_DIR / TEST_PATTERN.format(fold=fold)
    if not train_path.is_file() or not test_path.is_file():
        raise FileNotFoundError(f"Missing fold files: {train_path} or {test_path}")

    all_train = read_jsonl(train_path)
    test = read_jsonl(test_path)
    validate_records(all_train, TRAIN_TASKS, f"fold {fold} train", check_images)
    validate_records(test, DIRECT_TASKS, f"fold {fold} test", check_images)

    train, validation = grouped_train_validation_split(
        all_train, INTERNAL_VALIDATION_FRACTION, SEED + fold
    )
    train_images = {record["image_path"] for record in all_train}
    test_images = {record["image_path"] for record in test}
    overlap = train_images & test_images
    if overlap:
        raise ValueError(f"Fold {fold}: {len(overlap)} exact images occur in both train and test")

    optimization_train = make_balanced_training_epoch(train, SEED + 1000 + fold)
    return optimization_train, validation, test, all_train

## 5. Harmony image-text dataset and response-only collator

Only assistant response tokens contribute to the loss. Prompt and padding tokens are masked with -100. Images are inserted at the first user turn.

Qwen expands a high-resolution image into many visual tokens. The dataset therefore limits each CXR to 1,024 visual tokens before tokenization. Multimodal truncation is deliberately disabled because truncating only the combined token sequence can remove image placeholder tokens while retaining the full visual grid, producing an image-token mismatch. If a complete example still exceeds `MAX_LENGTH`, the loader raises an explicit error instead of silently corrupting the image/text alignment.

For batches larger than one, `input_ids`, masks, labels, and `mm_token_type_ids` are padded to the longest sequence. Qwen's flattened visual-patch tensors are concatenated across examples, while one `image_grid_thw` row is retained per image.

In [ ]:
def supervised_messages(record, include_assistant=True):
    messages = []
    image_inserted = False
    for message in record["messages"]:
        role = message["role"]
        if role == "assistant":
            continue
        content = []
        raw_content = message.get("content", "")
        if isinstance(raw_content, str):
            raw_content = [{"type": "text", "text": raw_content}]
        for item in raw_content:
            if isinstance(item, str):
                item = {"type": "text", "text": item}
            if item.get("type") == "text":
                if role == "user" and not image_inserted:
                    content.append({"type": "image"})
                    image_inserted = True
                content.append({"type": "text", "text": item["text"]})
        messages.append({"role": role, "content": content})
    if not image_inserted:
        raise ValueError(f"No user turn available for image in {record.get('id')}")
    if include_assistant:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": answer_text(record)}],
        })
    return messages


def load_cxr_with_pixel_budget(path):
    """Load a CXR and reduce its area without changing its aspect ratio."""
    with Image.open(path) as image_file:
        image = image_file.convert("RGB")

    width, height = image.size
    area = width * height
    if area <= MAX_IMAGE_PIXELS:
        return image

    scale = math.sqrt(MAX_IMAGE_PIXELS / area)
    resized_width = max(
        QWEN_VISION_PIXEL_FACTOR,
        int(width * scale) // QWEN_VISION_PIXEL_FACTOR * QWEN_VISION_PIXEL_FACTOR,
    )
    resized_height = max(
        QWEN_VISION_PIXEL_FACTOR,
        int(height * scale) // QWEN_VISION_PIXEL_FACTOR * QWEN_VISION_PIXEL_FACTOR,
    )
    return image.resize(
        (resized_width, resized_height),
        resample=Image.Resampling.LANCZOS,
    )


class HarmonyImageTextDataset(Dataset):
    def __init__(self, records, processor, max_length):
        self.records = records
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        prompt_messages = supervised_messages(record, include_assistant=False)
        full_messages = supervised_messages(record, include_assistant=True)
        prompt_text = render_chat_template(
            self.processor,
            prompt_messages,
            add_generation_prompt=True,
        )
        full_text = render_chat_template(
            self.processor,
            full_messages,
            add_generation_prompt=False,
        )

        image = load_cxr_with_pixel_budget(record["image_path"])
        full = self.processor(
            text=full_text,
            images=image,
            return_tensors="pt",
            truncation=False,
        )
        prompt = self.processor(
            text=prompt_text,
            images=image,
            return_tensors="pt",
            truncation=False,
        )

        full_length = int(full["input_ids"].shape[-1])
        prompt_length_unclipped = int(prompt["input_ids"].shape[-1])
        if full_length > self.max_length:
            raise ValueError(
                f"Complete sequence for {record['id']} has {full_length} tokens, "
                f"exceeding MAX_LENGTH={self.max_length}. Do not enable multimodal "
                "truncation. Reduce MAX_IMAGE_TOKENS or increase MAX_LENGTH."
            )
        if prompt_length_unclipped >= full_length:
            raise ValueError(
                f"No assistant target tokens remain for {record['id']}: "
                f"prompt={prompt_length_unclipped}, full={full_length}."
            )

        item = {key: value.squeeze(0) for key, value in full.items()}
        labels = item["input_ids"].clone()
        prompt_length = min(prompt["input_ids"].shape[-1], labels.shape[-1])
        labels[:prompt_length] = -100
        if torch.all(labels == -100):
            raise ValueError(
                f"Target was fully truncated for {record['id']}; increase MAX_LENGTH above {self.max_length}."
            )
        item["labels"] = labels
        return item


class MultiModalResponseOnlyCollator:
    def __init__(self, processor):
        self.processor = processor
        tokenizer = processor.tokenizer
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        self.pad_token_id = tokenizer.pad_token_id

    @staticmethod
    def _pad_sequence(tensor, target_length, value):
        if tensor.shape[0] == target_length:
            return tensor
        pad_shape = (target_length - tensor.shape[0],) + tuple(tensor.shape[1:])
        padding = torch.full(pad_shape, value, dtype=tensor.dtype)
        return torch.cat([tensor, padding], dim=0)

    def __call__(self, features):
        batch = {}
        sequence_keys = {
            "input_ids",
            "attention_mask",
            "token_type_ids",
            "mm_token_type_ids",
            "labels",
        }
        max_length = max(feature["input_ids"].shape[0] for feature in features)
        for key in sequence_keys:
            if all(key in feature for feature in features):
                value = -100 if key == "labels" else (self.pad_token_id if key == "input_ids" else 0)
                batch[key] = torch.stack([
                    self._pad_sequence(feature[key], max_length, value) for feature in features
                ])

        visual_patch_keys = {"pixel_values", "pixel_values_videos"}
        visual_grid_keys = {"image_grid_thw", "video_grid_thw"}

        common_keys = set.intersection(*(set(feature) for feature in features))
        for key in sorted(common_keys - sequence_keys):
            values = [feature[key] for feature in features]
            shapes = [tuple(value.shape) for value in values]
            try:
                if key in visual_patch_keys:
                    # Qwen stores a variable number of flattened visual patches
                    # for each image. The model expects all patches concatenated.
                    batch[key] = torch.cat(values, dim=0)
                elif key in visual_grid_keys:
                    # __getitem__ removes the singleton image dimension, so
                    # stacking restores one (t, h, w) grid row per image.
                    batch[key] = torch.stack(values, dim=0)
                else:
                    batch[key] = torch.stack(values, dim=0)
            except RuntimeError as exc:
                raise RuntimeError(
                    f"Cannot combine processor field {key}; shapes={shapes}. "
                    "Inspect the Qwen processor output for this field."
                ) from exc
        return batch

## 6. Load Qwen3.5 and attach rank-32 LoRA

Qwen3.5 mixes full-attention and linear-attention text layers. Restricting LoRA to only `q_proj/k_proj/v_proj/o_proj` would leave much of that hybrid backbone untouched. The target discovery below therefore selects linear modules throughout the language backbone while excluding the vision encoder, embeddings, output head, and any auxiliary prediction head.

The discovered targets and trainable-parameter count are printed for every fold and saved in the fold summary. This is important when comparing Qwen with MedGemma: equal LoRA rank does not guarantee an equal trainable-parameter budget.


In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_IMAGE_PIXELS,
    max_pixels=MAX_IMAGE_PIXELS,
)


VISION_NAME_MARKERS = (
    "visual",
    "vision_model",
    "vision_tower",
    "vision_encoder",
)
TEXT_NAME_MARKERS = (
    "language_model",
    "text_model",
    "model.layers",
)
LORA_EXCLUDE_MARKERS = (
    "lm_head",
    "embed_tokens",
    "embedding",
    "mtp",
)


def discover_qwen_text_lora_targets(model):
    targets = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue

        lower_name = name.lower()
        if any(marker in lower_name for marker in VISION_NAME_MARKERS):
            continue
        if any(marker in lower_name for marker in LORA_EXCLUDE_MARKERS):
            continue
        if any(marker in lower_name for marker in TEXT_NAME_MARKERS):
            targets.append(name)

    targets = sorted(set(targets))
    if not targets:
        sample_linear_names = [
            name for name, module in model.named_modules()
            if isinstance(module, torch.nn.Linear)
        ][:30]
        raise RuntimeError(
            "No Qwen text-backbone LoRA targets were discovered. "
            f"Example linear module names: {sample_linear_names}"
        )
    return targets


def build_lora_model():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "A CUDA GPU is required for practical Qwen3.5-4B LoRA training. "
            "The CPU dtype branch is retained for inspection/inference only."
        )

    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=MODEL_DTYPE,
        device_map="auto",
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    target_modules = discover_qwen_text_lora_targets(model)
    print(f"Discovered {len(target_modules)} Qwen text LoRA target modules")
    print("First targets:", target_modules[:20])

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=target_modules,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model._clinical_lora_target_names = target_modules
    model.print_trainable_parameters()
    return model


## 7. Generation-based metrics

Final evaluation generates the three direct outputs from held-out images. The code reports binary classification metrics and mRALE regression/component metrics. AUROC is intentionally omitted because greedy text generation does not provide a calibrated positive-class probability.

In [ ]:
def model_input_device(model):
    for parameter in model.parameters():
        if parameter.device.type not in {"meta", "cpu"}:
            return parameter.device
    return next(model.parameters()).device


def extract_json_object(text):
    cleaned = text.strip()
    cleaned = re.sub(r"^\`\`\`(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*\`\`\`$", "", cleaned)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    obj, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    return obj


@torch.inference_mode()
def generate_record(model, record):
    messages = supervised_messages(record, include_assistant=False)
    prompt_text = render_chat_template(
        processor,
        messages,
        add_generation_prompt=True,
    )
    image = load_cxr_with_pixel_budget(record["image_path"])
    inputs = processor(
        text=prompt_text,
        images=image,
        return_tensors="pt",
        truncation=False,
    )

    device = model_input_device(model)
    moved = {}
    for key, value in inputs.items():
        if value.is_floating_point():
            moved[key] = value.to(device=device, dtype=MODEL_DTYPE)
        else:
            moved[key] = value.to(device=device)
    input_length = moved["input_ids"].shape[-1]
    output_ids = model.generate(
        **moved,
        do_sample=False,
        max_new_tokens=GENERATION_MAX_NEW_TOKENS[record["task"]],
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
    return processor.decode(output_ids[0, input_length:], skip_special_tokens=True).strip()


def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else float("nan")


def binary_metrics(truths, predictions):
    valid = [(truth, pred) for truth, pred in zip(truths, predictions) if pred in {"Yes", "No"}]
    tp = sum(truth == "Yes" and pred == "Yes" for truth, pred in valid)
    tn = sum(truth == "No" and pred == "No" for truth, pred in valid)
    fp = sum(truth == "No" and pred == "Yes" for truth, pred in valid)
    fn = sum(truth == "Yes" and pred == "No" for truth, pred in valid)
    sensitivity = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    precision = safe_divide(tp, tp + fp)
    f1 = safe_divide(2 * tp, 2 * tp + fp + fn)
    denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return {
        "n": len(truths),
        "valid_predictions": len(valid),
        "valid_rate": safe_divide(len(valid), len(truths)),
        "accuracy": safe_divide(tp + tn, len(valid)),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "balanced_accuracy": (sensitivity + specificity) / 2 if not math.isnan(sensitivity) and not math.isnan(specificity) else float("nan"),
        "precision": precision,
        "f1": f1,
        "mcc": safe_divide(tp * tn - fp * fn, denominator),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def mrale_metrics(truth_objects, prediction_objects):
    valid_pairs = [(truth, pred) for truth, pred in zip(truth_objects, prediction_objects) if pred is not None]
    fields = [
        "extent_right_numerical",
        "density_right_numerical",
        "extent_left_numerical",
        "density_left_numerical",
    ]
    errors = []
    exact = 0
    component_correct = Counter()
    component_valid = Counter()
    formula_consistent = 0
    formula_valid = 0
    for truth, pred in valid_pairs:
        try:
            true_total = float(truth["mRALE Score"])
            pred_total = float(pred["mRALE Score"])
            errors.append(pred_total - true_total)
            exact += pred_total == true_total
        except (KeyError, TypeError, ValueError):
            continue
        for field in fields:
            try:
                component_valid[field] += 1
                component_correct[field] += float(pred[field]) == float(truth[field])
            except (KeyError, TypeError, ValueError):
                pass
        try:
            recomputed = (
                float(pred["extent_right_numerical"]) * float(pred["density_right_numerical"])
                + float(pred["extent_left_numerical"]) * float(pred["density_left_numerical"])
            )
            formula_valid += 1
            formula_consistent += recomputed == pred_total
        except (KeyError, TypeError, ValueError):
            pass

    metrics = {
        "n": len(truth_objects),
        "valid_json": len(valid_pairs),
        "valid_json_rate": safe_divide(len(valid_pairs), len(truth_objects)),
        "valid_numeric_total": len(errors),
        "mae": safe_divide(sum(abs(error) for error in errors), len(errors)),
        "rmse": math.sqrt(safe_divide(sum(error * error for error in errors), len(errors))) if errors else float("nan"),
        "exact_total_accuracy": safe_divide(exact, len(errors)),
        "formula_consistency": safe_divide(formula_consistent, formula_valid),
    }
    for field in fields:
        metrics[f"{field}_accuracy"] = safe_divide(component_correct[field], component_valid[field])
    return metrics


def evaluate_generation(model, records, output_path):
    model.eval()
    rows = []
    for index, record in enumerate(records, start=1):
        generated = generate_record(model, record)
        parsed = None
        parse_error = None
        try:
            parsed = extract_json_object(generated)
        except Exception as exc:
            parse_error = str(exc)
        rows.append({
            "id": record["id"],
            "task": record["task"],
            "image_path": record["image_path"],
            "ground_truth_text": answer_text(record),
            "generated_text": generated,
            "parsed_prediction": parsed,
            "parse_error": parse_error,
        })
        if index % 50 == 0 or index == len(records):
            print(f"Generated {index}/{len(records)}")

    with Path(output_path).open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    results = {}
    for task in sorted(DIRECT_TASKS):
        task_rows = [row for row in rows if row["task"] == task]
        if not task_rows:
            continue
        truths = [json.loads(row["ground_truth_text"]) for row in task_rows]
        predictions = [row["parsed_prediction"] for row in task_rows]
        if task == "covid_classification":
            results[task] = binary_metrics(
                [obj["covid_positive"] for obj in truths],
                [obj.get("covid_positive") if isinstance(obj, dict) else None for obj in predictions],
            )
        elif task == "normality_classification":
            results[task] = binary_metrics(
                [obj["normal"] for obj in truths],
                [obj.get("normal") if isinstance(obj, dict) else None for obj in predictions],
            )
        else:
            results[task] = mrale_metrics(truths, predictions)
    return results

## 8. Train and evaluate one fold

Checkpoints are resumable. The best checkpoint is selected by internal validation loss. The saved adapter directory contains LoRA weights only, plus the processor.

In [ ]:
def json_safe(value):
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


def write_trainer_history(history, path):
    keys = sorted({key for row in history for key in row})
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader()
        writer.writerows(history)


def train_fold(fold):
    print("=" * 80)
    print(f"FOLD {fold}")
    fold_dir = OUTPUT_ROOT / f"fold_{fold}"
    checkpoint_dir = fold_dir / "checkpoints"
    adapter_dir = fold_dir / "best_adapter"
    fold_dir.mkdir(parents=True, exist_ok=True)

    optimization_train, validation_records, test_records, original_train = load_fold(fold)
    describe_records("Original fold training file", original_train)
    describe_records("Optimization subset after internal split and rebalancing", optimization_train)
    describe_records("Internal validation", validation_records)
    describe_records("Held-out test", test_records)

    model = build_lora_model()
    train_dataset = HarmonyImageTextDataset(optimization_train, processor, MAX_LENGTH)
    validation_dataset = HarmonyImageTextDataset(validation_records, processor, MAX_LENGTH)
    test_dataset = HarmonyImageTextDataset(test_records, processor, MAX_LENGTH)
    collator = MultiModalResponseOnlyCollator(processor)

    # Validate variable-length text and visual fields before Trainer starts.
    collator_probe_count = min(4, len(train_dataset))
    collator_probe = collator([
        train_dataset[index] for index in range(collator_probe_count)
    ])
    print(
        "Collator preflight:",
        collator_probe_count,
        "examples;",
        {key: tuple(value.shape) for key, value in collator_probe.items()},
    )
    del collator_probe

    # Keep the model forward probe small so it does not distort GPU-memory use.
    probe = collator([train_dataset[0]])
    device = model_input_device(model)
    probe = {
        key: value.to(device=device, dtype=MODEL_DTYPE) if value.is_floating_point() else value.to(device)
        for key, value in probe.items()
    }
    with torch.no_grad():
        probe_loss = model(**probe).loss.item()
    print("Forward-pass probe loss:", probe_loss)
    del probe
    torch.cuda.empty_cache()

    training_args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=(MODEL_DTYPE == torch.bfloat16),
        fp16=(MODEL_DTYPE == torch.float16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch",
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        prediction_loss_only=True,
        report_to="none",
        seed=SEED + fold,
        data_seed=SEED + fold,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        data_collator=collator,
    )
    last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None
    if last_checkpoint:
        print("Resuming from:", last_checkpoint)
    trainer.train(resume_from_checkpoint=last_checkpoint)

    validation_loss_metrics = trainer.evaluate(validation_dataset, metric_key_prefix="validation")
    test_loss_metrics = trainer.evaluate(test_dataset, metric_key_prefix="test_teacher_forced")
    trainer.save_model(str(adapter_dir))
    processor.save_pretrained(str(adapter_dir))
    write_trainer_history(trainer.state.log_history, fold_dir / "trainer_history.csv")

    generation_metrics = {}
    if RUN_GENERATION_EVAL:
        generation_metrics = evaluate_generation(
            trainer.model,
            test_records,
            fold_dir / "test_predictions.jsonl",
        )

    summary = {
        "fold": fold,
        "model_id": MODEL_ID,
        "lora": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_module_count": len(getattr(model, "_clinical_lora_target_names", [])),
            "target_modules": getattr(model, "_clinical_lora_target_names", []),
        },
        "model_dtype": str(MODEL_DTYPE),
        "thinking_disabled": DISABLE_THINKING,

        "counts": {
            "original_train_records": len(original_train),
            "optimization_records": len(optimization_train),
            "internal_validation_records": len(validation_records),
            "test_records": len(test_records),
        },
        "validation_loss_metrics": validation_loss_metrics,
        "test_teacher_forced_metrics": test_loss_metrics,
        "generation_metrics": generation_metrics,
    }
    with (fold_dir / "fold_summary.json").open("w", encoding="utf-8") as handle:
        json.dump(json_safe(summary), handle, indent=2)

    del trainer, model, train_dataset, validation_dataset, test_dataset
    gc.collect()
    torch.cuda.empty_cache()
    return summary

## 9. Verify saved adapters and run only unfinished folds

Before training, this section verifies each requested fold under `OUTPUT_ROOT`.
A fold is considered trained only when `best_adapter/adapter_config.json` is valid
and a nonempty `adapter_model.safetensors` or `adapter_model.bin` is present.
Completed folds are skipped; if all requested folds pass verification, no fold
training is started. Each newly trained fold is verified again immediately after
saving, and the final status is recorded in `fold_adapter_completion_status.json`.

In [ ]:
ADAPTER_WEIGHT_FILENAMES = (
    "adapter_model.safetensors",
    "adapter_model.bin",
)


def inspect_saved_fold_adapter(fold: int) -> dict:
    """Return a detailed, conservative completion check for one saved LoRA."""
    fold_dir = OUTPUT_ROOT / f"fold_{fold}"
    adapter_dir = fold_dir / "best_adapter"
    config_path = adapter_dir / "adapter_config.json"
    weight_candidates = [adapter_dir / name for name in ADAPTER_WEIGHT_FILENAMES]
    usable_weights = [
        path for path in weight_candidates if path.is_file() and path.stat().st_size > 0
    ]

    status = {
        "fold": int(fold),
        "fold_dir": str(fold_dir),
        "adapter_dir": str(adapter_dir),
        "complete": False,
        "reason": "",
        "adapter_config": str(config_path),
        "adapter_weight": str(usable_weights[0]) if usable_weights else None,
        "adapter_weight_bytes": (
            int(usable_weights[0].stat().st_size) if usable_weights else 0
        ),
        "warning": None,
    }

    if not adapter_dir.is_dir():
        status["reason"] = "best_adapter directory is missing"
        return status
    if not config_path.is_file():
        status["reason"] = "adapter_config.json is missing"
        return status
    if not usable_weights:
        status["reason"] = "adapter weight file is missing or empty"
        return status

    try:
        with config_path.open("r", encoding="utf-8") as handle:
            adapter_config = json.load(handle)
    except (OSError, json.JSONDecodeError) as exc:
        status["reason"] = f"adapter_config.json is unreadable: {exc}"
        return status

    saved_rank = adapter_config.get("r")
    if saved_rank is not None:
        try:
            saved_rank = int(saved_rank)
        except (TypeError, ValueError):
            status["reason"] = f"adapter rank is invalid: {saved_rank!r}"
            return status
        if saved_rank != int(LORA_R):
            status["reason"] = (
                f"adapter rank mismatch: saved r={saved_rank}, expected r={LORA_R}"
            )
            return status

    saved_base_model = adapter_config.get("base_model_name_or_path")
    if saved_base_model and str(saved_base_model) != str(MODEL_ID):
        status["warning"] = (
            f"saved base model is {saved_base_model!r}; configured MODEL_ID is "
            f"{MODEL_ID!r}"
        )

    status["complete"] = True
    status["reason"] = "valid adapter configuration and nonempty adapter weights"
    return status


def print_fold_adapter_status(statuses: list[dict], heading: str) -> None:
    print(f"\n{heading}")
    for status in statuses:
        label = "DONE" if status["complete"] else "PENDING"
        print(
            f"  fold {status['fold']}: {label} - {status['reason']}\n"
            f"    {status['adapter_dir']}"
        )
        if status.get("adapter_weight"):
            size_mib = status["adapter_weight_bytes"] / (1024**2)
            print(f"    weights: {status['adapter_weight']} ({size_mib:.2f} MiB)")
        if status.get("warning"):
            print(f"    WARNING: {status['warning']}")


requested_folds = [int(fold) for fold in RUN_FOLDS]
initial_statuses = [inspect_saved_fold_adapter(fold) for fold in requested_folds]
print_fold_adapter_status(initial_statuses, "Saved LoRA verification before training")

pending_folds = [
    status["fold"] for status in initial_statuses if not status["complete"]
]

if not pending_folds:
    print(
        "\nAll requested folds already contain verified best_adapter LoRAs. "
        "Section 9 training is skipped."
    )
else:
    print(f"\nFolds requiring training or resume: {pending_folds}")
    for fold in requested_folds:
        pre_status = inspect_saved_fold_adapter(fold)
        if pre_status["complete"]:
            print(
                f"\nSkipping fold {fold}: verified adapter already exists at "
                f"{pre_status['adapter_dir']}"
            )
            continue

        print(f"\nStarting fold {fold}: {pre_status['reason']}")
        train_fold(fold)

        post_status = inspect_saved_fold_adapter(fold)
        if not post_status["complete"]:
            raise RuntimeError(
                f"Fold {fold} finished train_fold(), but its saved adapter failed "
                f"verification: {post_status['reason']}. Expected a valid LoRA at "
                f"{post_status['adapter_dir']}"
            )
        print(
            f"Verified fold {fold} adapter after training: "
            f"{post_status['adapter_weight']}"
        )

final_statuses = [inspect_saved_fold_adapter(fold) for fold in requested_folds]
print_fold_adapter_status(final_statuses, "Final saved LoRA verification")

completion_manifest = {
    "model_id": MODEL_ID,
    "expected_lora_rank": int(LORA_R),
    "requested_folds": requested_folds,
    "all_requested_folds_complete": all(
        status["complete"] for status in final_statuses
    ),
    "folds": final_statuses,
}
manifest_path = OUTPUT_ROOT / "fold_adapter_completion_status.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(completion_manifest, handle, indent=2)

incomplete = [
    status["fold"] for status in final_statuses if not status["complete"]
]
if incomplete:
    raise RuntimeError(
        f"The following folds still do not have verified best adapters: {incomplete}. "
        f"See {manifest_path} for details."
    )

# Rebuild this variable from saved summaries so it is useful whether folds were
# trained in this run or skipped because their adapters were already complete.
fold_summaries = []
missing_summary_folds = []
for fold in requested_folds:
    summary_path = OUTPUT_ROOT / f"fold_{fold}" / "fold_summary.json"
    if not summary_path.is_file():
        missing_summary_folds.append(fold)
        continue
    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)
    if int(summary.get("fold", fold)) != fold:
        raise ValueError(
            f"Fold summary identity mismatch in {summary_path}: "
            f"found fold={summary.get('fold')!r}, expected {fold}."
        )
    fold_summaries.append(summary)

print(f"\nVerified best adapters for folds: {requested_folds}")
print(f"Completion manifest: {manifest_path}")
print("Saved summaries available for folds:", [item["fold"] for item in fold_summaries])
if missing_summary_folds:
    print(
        "WARNING: verified adapters exist, but fold_summary.json is missing for "
        f"folds {missing_summary_folds}. Section 10 aggregation requires those "
        "summary files."
    )

## 10. Aggregate cross-validation metrics

This cell can be rerun independently after all folds finish. It reads the saved fold summaries, computes the mean, sample standard deviation, and two-sided 95% confidence interval across the five held-out folds, and writes both JSON and CSV summaries under `OUTPUT_ROOT`.

With only five folds, the confidence interval uses the Student-t critical value for four degrees of freedom. Fold-level intervals describe variability across this particular cross-validation split; they are not patient-level bootstrap intervals.


In [ ]:
# Local folder containing fold_0 through fold_4.
# Change this path if the result folder is copied to the cloud.
RESULTS_ROOT = OUTPUT_ROOT
EXPECTED_FOLDS = RUN_FOLDS
INCLUDE_COUNT_METRICS = False


def flatten_numeric(prefix, value, output):
    if isinstance(value, dict):
        for key, item in value.items():
            flatten_numeric(f"{prefix}.{key}" if prefix else key, item, output)
    elif isinstance(value, (int, float)) and not isinstance(value, bool):
        numeric_value = float(value)
        if math.isfinite(numeric_value):
            output[prefix] = numeric_value


def load_saved_fold_summaries(results_root, expected_folds):
    summaries = []
    for fold in expected_folds:
        summary_path = Path(results_root) / f"fold_{fold}" / "fold_summary.json"
        if not summary_path.is_file():
            raise FileNotFoundError(f"Missing fold summary: {summary_path}")
        with summary_path.open("r", encoding="utf-8") as handle:
            summary = json.load(handle)
        if int(summary.get("fold", -1)) != fold:
            raise ValueError(
                f"Fold mismatch in {summary_path}: expected {fold}, found {summary.get('fold')}"
            )
        summaries.append(summary)
    return summaries


def student_t_975(degrees_of_freedom):
    """Two-sided 95% critical values; normal approximation above 30 df."""
    critical_values = {
        1: 12.706205, 2: 4.302653, 3: 3.182446, 4: 2.776445,
        5: 2.570582, 6: 2.446912, 7: 2.364624, 8: 2.306004,
        9: 2.262157, 10: 2.228139, 11: 2.200985, 12: 2.178813,
        13: 2.160369, 14: 2.144787, 15: 2.131450, 16: 2.119905,
        17: 2.109816, 18: 2.100922, 19: 2.093024, 20: 2.085963,
        21: 2.079614, 22: 2.073873, 23: 2.068658, 24: 2.063899,
        25: 2.059539, 26: 2.055529, 27: 2.051831, 28: 2.048407,
        29: 2.045230, 30: 2.042272,
    }
    return critical_values.get(degrees_of_freedom, 1.959964)


def is_count_metric(metric_name):
    leaf = metric_name.rsplit(".", 1)[-1]
    return leaf in {
        "n", "valid_predictions", "valid_json", "valid_numeric_total",
        "tp", "tn", "fp", "fn", "epoch", "step",
    } or leaf.endswith("runtime") or leaf.endswith("samples_per_second") or leaf.endswith("steps_per_second")


saved_fold_summaries = load_saved_fold_summaries(RESULTS_ROOT, EXPECTED_FOLDS)

# One row per fold, including losses, generation metrics, and count fields.
per_fold_rows = []
for summary in saved_fold_summaries:
    row = {"fold": int(summary["fold"])}
    flatten_numeric("generation", summary.get("generation_metrics", {}), row)
    flatten_numeric(
        "loss",
        {
            "validation": summary.get("validation_loss_metrics", {}),
            "test_teacher_forced": summary.get("test_teacher_forced_metrics", {}),
        },
        row,
    )
    per_fold_rows.append(row)

all_keys = ["fold"] + sorted(
    {key for row in per_fold_rows for key in row if key != "fold"}
)
per_fold_csv = RESULTS_ROOT / "cross_validation_per_fold.csv"
with per_fold_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=all_keys)
    writer.writeheader()
    writer.writerows(per_fold_rows)

# Fold-level macro summary. CI = mean +/- t_(0.975, n-1) * sample_std / sqrt(n).
aggregate_rows = []
for metric in all_keys:
    if metric == "fold" or (not INCLUDE_COUNT_METRICS and is_count_metric(metric)):
        continue
    values = [row[metric] for row in per_fold_rows if metric in row and math.isfinite(row[metric])]
    n_folds = len(values)
    if not values:
        continue
    mean_value = sum(values) / n_folds
    if n_folds >= 2:
        sample_variance = sum((value - mean_value) ** 2 for value in values) / (n_folds - 1)
        sample_std = math.sqrt(sample_variance)
        t_critical = student_t_975(n_folds - 1)
        standard_error = sample_std / math.sqrt(n_folds)
        margin = t_critical * standard_error
        ci_lower = mean_value - margin
        ci_upper = mean_value + margin
    else:
        sample_std = float("nan")
        t_critical = float("nan")
        standard_error = float("nan")
        margin = float("nan")
        ci_lower = float("nan")
        ci_upper = float("nan")

    aggregate_rows.append({
        "metric": metric,
        "n_folds": n_folds,
        "mean": mean_value,
        "sample_std": sample_std,
        "standard_error": standard_error,
        "t_critical_95": t_critical,
        "ci95_margin": margin,
        "ci95_lower": ci_lower,
        "ci95_upper": ci_upper,
    })

aggregate_csv = RESULTS_ROOT / "cross_validation_aggregate_95ci.csv"
aggregate_fields = [
    "metric", "n_folds", "mean", "sample_std", "standard_error",
    "t_critical_95", "ci95_margin", "ci95_lower", "ci95_upper",
]
with aggregate_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=aggregate_fields)
    writer.writeheader()
    writer.writerows(aggregate_rows)

aggregate_json = RESULTS_ROOT / "cross_validation_aggregate_95ci.json"
with aggregate_json.open("w", encoding="utf-8") as handle:
    json.dump(json_safe({row["metric"]: row for row in aggregate_rows}), handle, indent=2)

print(f"Loaded {len(saved_fold_summaries)} folds from {RESULTS_ROOT}")
print(f"Per-fold metrics: {per_fold_csv}")
print(f"Aggregate metrics: {aggregate_csv}")
print()
for row in aggregate_rows:
    print(
        f"{row['metric']}: mean={row['mean']:.6f}, "
        f"SD={row['sample_std']:.6f}, "
        f"95% CI=[{row['ci95_lower']:.6f}, {row['ci95_upper']:.6f}]"
    )

## 11. Reporting cautions

- Compare Qwen3.5 with MedGemma using the identical fold files and the same held-out metrics.
- Report each model's trainable-parameter count as well as LoRA rank. Qwen's hybrid architecture means rank 32 alone does not define an equal adaptation budget.
- Keep Qwen thinking disabled for direct structured JSON evaluation. If reasoning traces are studied later, treat that as a separate experiment with its own targets and parser.
- Report COVID classification as PCR-status prediction within MIDRC and explicitly acknowledge the dataset/source confounding introduced by Montgomery negatives.
- Report mRALE performance primarily on MIDRC, where mRALE labels exist.
- Report normality performance primarily on Montgomery, where normality labels originate.
- Do not describe AI-generated MIMIC-Ext bounding boxes as human ground truth or use them as clinical localization validation.
- Use an independent external cohort, preferably with appropriate human labels, before making clinical performance claims.
